# SimpleLLM V0.24.1 — Anti-Overfitting + Massive Corpus

## Diagnosis (from V0.21 training)

| Evidence | Root Cause |
|----------|-----------|
| "Adam, the son of Hur" — verbatim Bible | Bible KJV = 20% of V0.21 corpus |
| Gutenberg license text in output | Weak boilerplate filter |
| All prompts → biblical style | Dataset imbalance + overtraining |

## Changes from V0.21

| Fix | V0.21 | V0.24.1 |
|-----|-------|---------|
| **Corpus** | 33 books, ~160K lines, ~4M tokens | **80+ books, ~700K lines, ~20M tokens** |
| Dataset cap | None (Bible dominated) | **Removed — 80+ diverse books, no single source >8%** |
| Gutenberg filter | Keyword "gutenberg" only | **25+ blacklist patterns** |
| Vocab size | 8,000 BPE | **12,000 BPE** (bigger corpus needs bigger vocab) |
| Dropout | 0.15 | **0.20** |
| Label smoothing | None | **0.1** |
| Weight decay | 0 (Adam) | **0.01 (AdamW)** |
| Epochs | 30 | **20** + EarlyStopping patience=3 |
| Repetition penalty | None | **1.2** in generate() |
| Monitoring | None | **OverfittingDetector, GenerationSampler** |


In [ ]:
# ========================== [CELL 1] ЗАВИСИМОСТИ ==========================
!pip install -q sentencepiece

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks
import sentencepiece as spm
import os, urllib.request, tempfile, time, re, math

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {gpus}")


In [ ]:
# ========================== [CELL 2] ГИПЕРПАРАМЕТРЫ ==========================
#
# ИЗМЕНЕНИЯ V0.24.1 (масштабирование данных):
# - VOCAB_SIZE: 8000 -> 12000  (больше корпус = нужен больший словарь BPE)
# - MAX_LINES_PER_DATASET: 5000 -> 0 (нет лимита — 80+ книг, баланс не нужен)
# - MAX_TRAIN_LINES: 300K -> 1M  (корпус вырос 5-7x)
# - EPOCHS: 25 -> 20  (больше данных = меньше эпох, EarlyStopping patience=3)
# - Остальные anti-overfitting фиксы без изменений.

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Vocabulary / Tokenizer ----
VOCAB_SIZE = 12000           # 8K -> 12K: больший корпус → нужен больший BPE vocab
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

# ---- Model Architecture ----
CONTEXT_WIN = 128
EMBED_DIM = 256
NUM_HEADS = 8
FF_DIM = 1024
NUM_BLOCKS = 6
DROPOUT_RATE = 0.20

# ---- Training ----
BATCH_SIZE = 64
EPOCHS = 20                  # 25 -> 20: больше данных = меньше проходов
LEARNING_RATE = 5e-4
MIN_LR = 1e-5
WARMUP_STEPS = 1500          # 1000 -> 1500: больше steps/epoch
VALIDATION_SPLIT = 0.1

# ---- Data ----
MAX_TRAIN_LINES = 1_000_000  # 300K -> 1M: корпус вырос
MIN_LINE_LENGTH = 15
MAX_LINES_PER_DATASET = 0    # 0 = без лимита. 80+ книг — ни одна не доминирует.

# ---- Anti-overfitting ----
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 0.01

# ---- Mixed Precision ----
USE_MIXED_PRECISION = True
if USE_MIXED_PRECISION and len(gpus) > 0:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision: ENABLED")
else:
    print("Mixed precision: DISABLED")

estimated_params = (VOCAB_SIZE * EMBED_DIM +
                    NUM_BLOCKS * (4 * EMBED_DIM**2 + 2 * EMBED_DIM * FF_DIM))
print(f"Ожидаемые параметры: ~{estimated_params / 1e6:.1f}M")
print(f"\nV0.24.1 Config:")
print(f"  Vocab size:       {VOCAB_SIZE}")
print(f"  Dropout:          {DROPOUT_RATE}")
print(f"  Label smoothing:  {LABEL_SMOOTHING}")
print(f"  Weight decay:     {WEIGHT_DECAY}")
print(f"  Max lines/dataset:{MAX_LINES_PER_DATASET} (0=unlimited)")
print(f"  Max train lines:  {MAX_TRAIN_LINES:,}")
print(f"  Epochs:           {EPOCHS}")
print(f"  EarlyStopping:    patience=3")


In [ ]:
# ========================== [CELL 3] ЗАГРУЗКА ДАННЫХ ==========================
#
# V0.24.1: 80+ книг из Project Gutenberg + Shakespeare + Bible.
# Цель: ~600-800K строк → ~20-25M BPE токенов.
# MAX_LINES_PER_DATASET = 0 (без ограничений — корпус достаточно разнообразен).
#
# Категории:
#   MEGA романы (40K+ строк): Les Mis, War & Peace, Don Quixote, etc.
#   Классика (10-30K): Dickens, Austen, Brontë, Twain, Dostoevsky, etc.
#   Средние (5-10K): Wells, Kipling, London, Stevenson, etc.
#   Малые (<5K): Jekyll & Hyde, Oz, Christmas Carol — для жанрового разнообразия.
#   Нон-фикшн: Wealth of Nations, Federalist Papers, Walden, On Liberty, etc.
#
# Удалено: tiny_shakespeare (дубликат shakespeare)

def _g(epub_id):
    """Shortcut: Gutenberg URL by epub ID."""
    return f'https://www.gutenberg.org/cache/epub/{epub_id}/pg{epub_id}.txt'

DATASETS = {

    # ===== НЕ-GUTENBERG =====
    'shakespeare': {
        'url': 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt',
        'filename': 'shakespeare.txt'
    },
    'bible_kjv': {
        'url': 'https://raw.githubusercontent.com/mxw/grmr/master/src/finaltests/bible.txt',
        'filename': 'bible_kjv.txt'
    },

    # ===== MEGA РОМАНЫ (40K+ строк — основные контрибуторы) =====
    'les_miserables':           {'url': _g(135),   'filename': 'les_miserables.txt'},
    'war_and_peace':            {'url': _g(2600),  'filename': 'war_and_peace.txt'},
    'don_quixote_1':            {'url': _g(996),   'filename': 'don_quixote_1.txt'},
    'don_quixote_2':            {'url': _g(5946),  'filename': 'don_quixote_2.txt'},
    'brothers_karamazov':       {'url': _g(28054), 'filename': 'brothers_karamazov.txt'},
    'ulysses':                  {'url': _g(4300),  'filename': 'ulysses.txt'},
    'monte_cristo':             {'url': _g(1184),  'filename': 'monte_cristo.txt'},

    # ===== DICKENS (каждый роман 15-35K строк) =====
    'david_copperfield':        {'url': _g(766),   'filename': 'david_copperfield.txt'},
    'bleak_house':              {'url': _g(1023),  'filename': 'bleak_house.txt'},
    'great_expectations':       {'url': _g(1400),  'filename': 'great_expectations.txt'},
    'tale_two_cities':          {'url': _g(98),    'filename': 'tale_two_cities.txt'},
    'oliver_twist':             {'url': _g(730),   'filename': 'oliver_twist.txt'},
    'nicholas_nickleby':        {'url': _g(967),   'filename': 'nicholas_nickleby.txt'},
    'pickwick_papers':          {'url': _g(580),   'filename': 'pickwick_papers.txt'},
    'our_mutual_friend':        {'url': _g(883),   'filename': 'our_mutual_friend.txt'},
    'little_dorrit':            {'url': _g(963),   'filename': 'little_dorrit.txt'},
    'christmas_carol':          {'url': _g(46),    'filename': 'christmas_carol.txt'},

    # ===== AUSTEN =====
    'pride_prejudice':          {'url': _g(1342),  'filename': 'pride_prejudice.txt'},
    'emma':                     {'url': _g(158),   'filename': 'emma.txt'},
    'sense_sensibility':        {'url': _g(161),   'filename': 'sense_sensibility.txt'},
    'mansfield_park':           {'url': _g(141),   'filename': 'mansfield_park.txt'},
    'persuasion':               {'url': _g(105),   'filename': 'persuasion.txt'},
    'northanger_abbey':         {'url': _g(121),   'filename': 'northanger_abbey.txt'},

    # ===== BRONTË =====
    'jane_eyre':                {'url': _g(1260),  'filename': 'jane_eyre.txt'},
    'wuthering_heights':        {'url': _g(768),   'filename': 'wuthering_heights.txt'},

    # ===== РУССКИЕ КЛАССИКИ (англ. переводы) =====
    'crime_punishment':         {'url': _g(2554),  'filename': 'crime_punishment.txt'},
    'anna_karenina':            {'url': _g(1399),  'filename': 'anna_karenina.txt'},

    # ===== ФРАНЦУЗСКИЕ КЛАССИКИ (англ. переводы) =====
    'three_musketeers':         {'url': _g(1257),  'filename': 'three_musketeers.txt'},
    'hunchback_notre_dame':     {'url': _g(2610),  'filename': 'hunchback_notre_dame.txt'},
    'phantom_opera':            {'url': _g(175),   'filename': 'phantom_opera.txt'},

    # ===== MARK TWAIN =====
    'tom_sawyer':               {'url': _g(74),    'filename': 'tom_sawyer.txt'},
    'huck_finn':                {'url': _g(76),    'filename': 'huck_finn.txt'},
    'connecticut_yankee':       {'url': _g(86),    'filename': 'connecticut_yankee.txt'},
    'life_on_mississippi':      {'url': _g(245),   'filename': 'life_on_mississippi.txt'},
    'innocents_abroad':         {'url': _g(3176),  'filename': 'innocents_abroad.txt'},
    'prince_and_pauper':        {'url': _g(1837),  'filename': 'prince_and_pauper.txt'},

    # ===== HERMAN MELVILLE =====
    'moby_dick':                {'url': _g(2701),  'filename': 'moby_dick.txt'},

    # ===== GEORGE ELIOT =====
    'middlemarch':              {'url': _g(145),   'filename': 'middlemarch.txt'},

    # ===== THACKERAY =====
    'vanity_fair':              {'url': _g(599),   'filename': 'vanity_fair.txt'},

    # ===== ПРИКЛЮЧЕНИЯ =====
    'treasure_island':          {'url': _g(120),   'filename': 'treasure_island.txt'},
    'kidnapped':                {'url': _g(421),   'filename': 'kidnapped.txt'},
    'king_solomons_mines':      {'url': _g(2166),  'filename': 'king_solomons_mines.txt'},
    'jungle_book':              {'url': _g(236),   'filename': 'jungle_book.txt'},
    'robinson_crusoe':          {'url': _g(521),   'filename': 'robinson_crusoe.txt'},
    'gulliver':                 {'url': _g(829),   'filename': 'gulliver.txt'},
    'around_the_world':         {'url': _g(103),   'filename': 'around_the_world.txt'},
    'twenty_thousand_leagues':  {'url': _g(164),   'filename': 'twenty_thousand_leagues.txt'},
    'ivanhoe':                  {'url': _g(82),    'filename': 'ivanhoe.txt'},
    'last_of_mohicans':         {'url': _g(940),   'filename': 'last_of_mohicans.txt'},
    'scarlet_pimpernel':        {'url': _g(60),    'filename': 'scarlet_pimpernel.txt'},
    'tarzan':                   {'url': _g(78),    'filename': 'tarzan.txt'},
    'call_of_wild':             {'url': _g(215),   'filename': 'call_of_wild.txt'},
    'white_fang':               {'url': _g(910),   'filename': 'white_fang.txt'},

    # ===== НАУЧНАЯ ФАНТАСТИКА =====
    'war_of_worlds':            {'url': _g(36),    'filename': 'war_of_worlds.txt'},
    'time_machine':             {'url': _g(35),    'filename': 'time_machine.txt'},
    'invisible_man':            {'url': _g(5230),  'filename': 'invisible_man.txt'},
    'island_dr_moreau':         {'url': _g(159),   'filename': 'island_dr_moreau.txt'},
    'lost_world':               {'url': _g(139),   'filename': 'lost_world.txt'},
    'princess_mars':            {'url': _g(62),    'filename': 'princess_mars.txt'},

    # ===== ГОТИКА / ХОРРОР =====
    'frankenstein':             {'url': _g(84),    'filename': 'frankenstein.txt'},
    'dracula':                  {'url': _g(345),   'filename': 'dracula.txt'},
    'jekyll_hyde':              {'url': _g(43),    'filename': 'jekyll_hyde.txt'},
    'turn_of_screw':            {'url': _g(209),   'filename': 'turn_of_screw.txt'},
    'dorian_gray':              {'url': _g(174),   'filename': 'dorian_gray.txt'},

    # ===== ДЕТЕКТИВ =====
    'sherlock_adventures':      {'url': _g(1661),  'filename': 'sherlock_adventures.txt'},
    'sherlock_memoirs':         {'url': _g(834),   'filename': 'sherlock_memoirs.txt'},
    'sherlock_return':          {'url': _g(108),   'filename': 'sherlock_return.txt'},
    'hound_baskervilles':       {'url': _g(2852),  'filename': 'hound_baskervilles.txt'},
    'study_in_scarlet':         {'url': _g(244),   'filename': 'study_in_scarlet.txt'},
    'sign_of_four':             {'url': _g(2097),  'filename': 'sign_of_four.txt'},
    'valley_of_fear':           {'url': _g(3289),  'filename': 'valley_of_fear.txt'},
    'moonstone':                {'url': _g(155),   'filename': 'moonstone.txt'},
    'woman_in_white':           {'url': _g(583),   'filename': 'woman_in_white.txt'},

    # ===== ИРЛАНДСКАЯ / МОДЕРНИЗМ =====
    'portrait_artist':          {'url': _g(4217),  'filename': 'portrait_artist.txt'},
    'dubliners':                {'url': _g(2814),  'filename': 'dubliners.txt'},

    # ===== АМЕРИКАНСКАЯ ЛИТЕРАТУРА =====
    'little_women':             {'url': _g(514),   'filename': 'little_women.txt'},
    'scarlet_letter':           {'url': _g(25344), 'filename': 'scarlet_letter.txt'},
    'house_seven_gables':       {'url': _g(77),    'filename': 'house_seven_gables.txt'},
    'age_of_innocence':         {'url': _g(541),   'filename': 'age_of_innocence.txt'},
    'secret_garden':            {'url': _g(113),   'filename': 'secret_garden.txt'},
    'anne_green_gables':        {'url': _g(45),    'filename': 'anne_green_gables.txt'},
    'wizard_of_oz':             {'url': _g(55),    'filename': 'wizard_of_oz.txt'},

    # ===== ДЕТСКАЯ ЛИТЕРАТУРА / ФЭНТЕЗИ =====
    'alice':                    {'url': _g(11),    'filename': 'alice.txt'},
    'peter_pan':                {'url': _g(16),    'filename': 'peter_pan.txt'},
    'wind_in_willows':          {'url': _g(289),   'filename': 'wind_in_willows.txt'},

    # ===== ПЬЕСЫ / ДРАМАТУРГИЯ =====
    'importance_earnest':       {'url': _g(844),   'filename': 'importance_earnest.txt'},
    'pygmalion':                {'url': _g(3825),  'filename': 'pygmalion.txt'},

    # ===== НОН-ФИКШН / ФИЛОСОФИЯ / ЭКОНОМИКА =====
    'plato_republic':           {'url': _g(1497),  'filename': 'plato_republic.txt'},
    'darwin_origin':            {'url': _g(1228),  'filename': 'darwin_origin.txt'},
    'einstein_relativity':      {'url': _g(5001),  'filename': 'einstein_relativity.txt'},
    'the_prince':               {'url': _g(1232),  'filename': 'the_prince.txt'},
    'art_of_war':               {'url': _g(132),   'filename': 'art_of_war.txt'},
    'utopia':                   {'url': _g(2130),  'filename': 'utopia.txt'},
    'wealth_of_nations':        {'url': _g(3300),  'filename': 'wealth_of_nations.txt'},
    'on_liberty':               {'url': _g(34901), 'filename': 'on_liberty.txt'},
    'leviathan':                {'url': _g(3207),  'filename': 'leviathan.txt'},
    'walden':                   {'url': _g(205),   'filename': 'walden.txt'},
    'federalist_papers':        {'url': _g(1404),  'filename': 'federalist_papers.txt'},
    'franklin_autobiography':   {'url': _g(20203), 'filename': 'franklin_autobiography.txt'},

    # ===== МЕМУАРЫ / АВТОБИОГРАФИИ =====
    'up_from_slavery':          {'url': _g(2376),  'filename': 'up_from_slavery.txt'},
    'frederick_douglass':       {'url': _g(23),    'filename': 'frederick_douglass.txt'},
    'souls_black_folk':         {'url': _g(408),   'filename': 'souls_black_folk.txt'},

    # ===== ПОЭЗИЯ (эпическая — длинные тексты) =====
    'odyssey':                  {'url': _g(1727),  'filename': 'odyssey.txt'},
    'iliad':                    {'url': _g(6130),  'filename': 'iliad.txt'},
    'poe':                      {'url': _g(2147),  'filename': 'poe.txt'},
    'heart_of_darkness':        {'url': _g(219),   'filename': 'heart_of_darkness.txt'},
}

print(f"Всего датасетов в конфиге: {len(DATASETS)}")


def clean_line(line):
    """Очищает строку: Unicode -> ASCII, множественные пробелы -> один."""
    line = line.strip()
    line = line.replace('\u2018', "'").replace('\u2019', "'")
    line = line.replace('\u201c', '"').replace('\u201d', '"')
    line = line.replace('\u2014', '--').replace('\u2013', '-')
    line = line.replace('\u2026', '...')
    line = re.sub(r'[^\x20-\x7E]', '', line)
    line = re.sub(r'\s+', ' ', line).strip()
    return line


# Расширенный фильтр Gutenberg boilerplate
GUTENBERG_BLACKLIST = [
    'gutenberg',
    'project gutenberg',
    'full refund',
    'electronic work',
    'paragraph 1.',
    'united states copyright',
    'literary archive',
    'donation',
    'public domain',
    'distribute copies',
    'copyright holder',
    'trademark',
    'ebook',
    'e-book',
    'redistribut',
    'proofreading',
    'section 1.',
    'section 2.',
    'section 3.',
    'section 4.',
    'section 5.',
    '501(c)(3)',
    'tax exempt',
    'www.gutenberg.org',
    'agreement',
]

def is_gutenberg_boilerplate(line_lower):
    """Проверяет является ли строка частью Gutenberg boilerplate."""
    return any(pattern in line_lower for pattern in GUTENBERG_BLACKLIST)


# ---- Загрузка и фильтрация ----
cache_dir = os.path.join(os.path.expanduser('~'), '.keras', 'datasets', 'llm_corpus_v024')
os.makedirs(cache_dir, exist_ok=True)

all_lines = []
dataset_stats = {}
lines_before_cap = {}

failed = []
for name, info in DATASETS.items():
    filepath = os.path.join(cache_dir, info['filename'])
    try:
        if not os.path.exists(filepath):
            print(f"  Скачиваю {name}...", end=' ')
            urllib.request.urlretrieve(info['url'], filepath)
            print("OK")

        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.read().split('\n')

        filtered = []
        for line in lines:
            cleaned = clean_line(line)
            lower = cleaned.lower()
            if (len(cleaned) > MIN_LINE_LENGTH
                and not cleaned.startswith('***')
                and not is_gutenberg_boilerplate(lower)):
                filtered.append(cleaned)

        lines_before_cap[name] = len(filtered)

        # Балансировка: если MAX_LINES_PER_DATASET > 0, ограничиваем
        if MAX_LINES_PER_DATASET > 0 and len(filtered) > MAX_LINES_PER_DATASET:
            rng = np.random.RandomState(SEED)
            indices = rng.choice(len(filtered), size=MAX_LINES_PER_DATASET, replace=False)
            filtered = [filtered[i] for i in sorted(indices)]

        all_lines.extend(filtered)
        dataset_stats[name] = len(filtered)
        print(f"  OK {name}: {len(filtered):,} строк")

    except Exception as e:
        failed.append(name)
        print(f"  FAIL {name}: {e}")

np.random.shuffle(all_lines)
train_text = [line + ' endseq' for line in all_lines[:MAX_TRAIN_LINES]]

assert len(train_text) > 10000, f"Корпус слишком маленький: {len(train_text)}"

total_chars = sum(len(line) for line in train_text)
est_tokens = total_chars // 4  # ~4 символа на BPE токен

print(f"\n{'='*65}")
print(f"  Датасетов загружено:      {len(dataset_stats)}/{len(DATASETS)}"
      f" ({len(failed)} failed)")
print(f"  Всего строк в корпусе:    {len(all_lines):,}")
print(f"  Строк для обучения:       {len(train_text):,}")
print(f"  Символов:                 {total_chars:,}")
print(f"  ~Токенов (оценка):        {est_tokens:,}")
print(f"  Ratio tokens/params:      ~{est_tokens / estimated_params:.1f}x")
print(f"  MAX_LINES_PER_DATASET:    {'unlimited' if MAX_LINES_PER_DATASET == 0 else MAX_LINES_PER_DATASET}")
print(f"{'='*65}")

if failed:
    print(f"\n  FAILED ({len(failed)}): {', '.join(failed)}")

# Top-10 крупнейших датасетов
print(f"\n  Top-10 датасетов по размеру:")
for name, count in sorted(dataset_stats.items(), key=lambda x: -x[1])[:10]:
    pct = count / len(all_lines) * 100
    print(f"    {name:30s}: {count:>8,} строк ({pct:.1f}%)")

# Длины строк
lengths = [len(line) for line in train_text]
print(f"\n  Длина строк: min={min(lengths)}, max={max(lengths)}, "
      f"mean={np.mean(lengths):.0f}, median={np.median(lengths):.0f}")


In [ ]:
# ========================== [CELL 4] SENTENCEPIECE BPE ==========================
# Тренирует BPE токенизатор на корпусе ~700K строк (80+ книг).
# V0.24.1: input_sentence_size=200K (было 50K), VOCAB_SIZE=12K (было 8K).

SP_MODEL_PREFIX = 'spm_bpe_v024'

# Создаём trainfile: одна строка = один документ
trainfile = os.path.join(cache_dir, f'{SP_MODEL_PREFIX}_traindata.txt')
with open(trainfile, 'w', encoding='utf-8') as f:
    for line in train_text:
        # Ограничиваем длину строки 512 символами (SentencePiece зависает на длинных)
        f.write(line[:512] + '\n')

print(f"Файл для SP: {trainfile}")
print(f"  Строк: {len(train_text):,}")

# Тренировка SentencePiece BPE
spm.SentencePieceTrainer.train(
    input=trainfile,
    model_prefix=SP_MODEL_PREFIX,
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    pad_id=PAD_ID,
    unk_id=UNK_ID,
    bos_id=BOS_ID,
    eos_id=EOS_ID,
    pad_piece='<pad>',
    unk_piece='<unk>',
    bos_piece='<s>',
    eos_piece='</s>',
    byte_fallback=True,
    max_sentence_length=512,
    input_sentence_size=200000,   # Увеличено: корпус ~700K строк
    shuffle_input_sentence=True,
    num_threads=os.cpu_count() or 4,
    train_extremely_large_corpus=False,
)

sp_model = spm.SentencePieceProcessor()
sp_model.load(f'{SP_MODEL_PREFIX}.model')

assert sp_model.get_piece_size() == VOCAB_SIZE
print(f"\nSP модель загружена: {sp_model.get_piece_size()} токенов")
print(f"  PAD={sp_model.pad_id()}, UNK={sp_model.unk_id()}, "
      f"BOS={sp_model.bos_id()}, EOS={sp_model.eos_id()}")

# Тест encode/decode
test_sent = "The quick brown fox jumps over the lazy dog."
encoded = sp_model.encode(test_sent)
decoded = sp_model.decode(encoded)
print(f"\n  Test encode: '{test_sent}'")
print(f"  -> ids ({len(encoded)}): {encoded[:20]}...")
print(f"  -> decode: '{decoded}'")


In [ ]:
# ========================== [CELL 5] TOKENIZER WRAPPER ==========================
# SPTokenizer: encode/decode + batch операции.
# Без изменений от V0.21.

class SPTokenizer:
    """Обёртка над SentencePieceProcessor для удобного encode/decode."""

    def __init__(self, sp_model):
        self.sp = sp_model
        self.vocab_size = sp_model.get_piece_size()
        self.pad_id = sp_model.pad_id()
        self.unk_id = sp_model.unk_id()
        self.bos_id = sp_model.bos_id()
        self.eos_id = sp_model.eos_id()

    def encode(self, text, add_bos=True, add_eos=True, max_len=None):
        ids = self.sp.encode(text)
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        if max_len is not None and len(ids) > max_len:
            ids = ids[:max_len]
        return ids

    def decode(self, ids):
        # Убираем спец-токены  before decoding
        clean = [i for i in ids if i not in (self.pad_id, self.bos_id, self.eos_id)]
        return self.sp.decode(clean)

    def encode_batch(self, texts, max_len=None, add_bos=True, add_eos=True):
        """Пакетное кодирование с padding (RIGHT-pad)."""
        if max_len is None:
            max_len = CONTEXT_WIN

        # Чанкованное кодирование: 5000 строк за раз (SP зависает на >50K)
        CHUNK = 5000
        all_encoded = []
        for start in range(0, len(texts), CHUNK):
            chunk = texts[start:start + CHUNK]
            for text in chunk:
                ids = self.encode(text, add_bos=add_bos, add_eos=add_eos, max_len=max_len)
                all_encoded.append(ids)

        # Pad to max_len (RIGHT-pad)
        padded = np.full((len(all_encoded), max_len), self.pad_id, dtype=np.int32)
        for i, ids in enumerate(all_encoded):
            padded[i, :len(ids)] = ids
        return padded

    def __repr__(self):
        return f"SPTokenizer(vocab={self.vocab_size}, pad={self.pad_id})"


tokenizer = SPTokenizer(sp_model)
print(tokenizer)

# Smoke test
test_batch = tokenizer.encode_batch(["Hello world", "The cat sat on the mat"], max_len=20)
print(f"  Batch shape: {test_batch.shape}")
print(f"  Sample: {test_batch[0].tolist()}")


In [ ]:
# ========================== [CELL 6] TF.DATA PIPELINE ==========================
# Формирует input/target пары с sample_weight маской для PAD.
# Без изменений от V0.21.

print("Токенизация корпуса...")
data = tokenizer.encode_batch(train_text, max_len=CONTEXT_WIN + 1)
print(f"  Tokenized shape: {data.shape}")

# Input = tokens[:-1], Target = tokens[1:]
input_data = data[:, :-1]   # (N, CONTEXT_WIN)
target_data = data[:, 1:]   # (N, CONTEXT_WIN)

# sample_weight: 1.0 для не-PAD, 0.0 для PAD
#   Это критический фикс V0.21: без этого модель учится предсказывать PAD
#   (а PAD = самый частый токен → loss падает, но модель ничего не учит)
sample_weight = (target_data != tokenizer.pad_id).astype(np.float32)

print(f"  input_data:    {input_data.shape}")
print(f"  target_data:   {target_data.shape}")
print(f"  sample_weight: {sample_weight.shape}")
print(f"  Non-PAD tokens: {sample_weight.sum():.0f} / {sample_weight.size} "
      f"({sample_weight.mean()*100:.1f}%)")

# Train / Val split (90/10)
N = len(input_data)
split = int(N * 0.9)
indices = np.random.permutation(N)

train_idx = indices[:split]
val_idx = indices[split:]

train_ds = tf.data.Dataset.from_tensor_slices((
    input_data[train_idx],
    target_data[train_idx],
    sample_weight[train_idx],
)).shuffle(10000, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((
    input_data[val_idx],
    target_data[val_idx],
    sample_weight[val_idx],
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"\n  Train: {len(train_idx):,} samples, ~{len(train_idx)//BATCH_SIZE} batches")
print(f"  Val:   {len(val_idx):,} samples, ~{len(val_idx)//BATCH_SIZE} batches")

# Sanity check: подсмотрим один батч
for x, y, w in train_ds.take(1):
    print(f"\n  Batch shapes: x={x.shape}, y={y.shape}, w={w.shape}")
    print(f"  x[0][:15] = {x[0][:15].numpy().tolist()}")
    print(f"  y[0][:15] = {y[0][:15].numpy().tolist()}")
    print(f"  w[0][:15] = {w[0][:15].numpy().tolist()}")


In [ ]:
# ========================== [CELL 7] CUSTOM METRICS + EMBEDDING ==========================
# Perplexity metric (label-smoothing aware) + TokenPositionEmbedding.
#
# ФИКС V0.24: Perplexity metric использует from_logits=True с label_smoothing=0
#   для точного подсчёта (label smoothing в loss — отдельная тема).
# ФИКС V0.21->V0.24: scale cast к token_emb.dtype (mixed precision compatibility).

class Perplexity(tf.keras.metrics.Mean):
    """Masked perplexity: exp(cross-entropy) только по не-PAD токенам."""

    def __init__(self, name='perplexity', **kwargs):
        super().__init__(name=name, **kwargs)
        # Перплексию считаем БЕЗ label smoothing (иначе цифры не интерпретируемы)
        self._xent = tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True, reduction='none')

    def update_state(self, y_true, y_pred, sample_weight=None):
        loss_per_tok = self._xent(y_true, y_pred)  # (B, T)
        if sample_weight is not None:
            loss_per_tok *= sample_weight
            total_loss = tf.reduce_sum(loss_per_tok)
            count = tf.reduce_sum(sample_weight) + 1e-8
            avg_loss = total_loss / count
        else:
            avg_loss = tf.reduce_mean(loss_per_tok)
        perp = tf.exp(tf.minimum(avg_loss, 20.0))  # cap при exp(20) ≈ 485M
        super().update_state(perp)

    def get_config(self):
        return super().get_config()


class TokenPositionEmbedding(tf.keras.layers.Layer):
    """Token embedding + позиционный embedding.  Weight tying: .token_emb используется в output."""

    def __init__(self, vocab_size, embed_dim, context_win, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.context_win = context_win
        self.token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim, name='tok_emb')
        self.pos_emb = tf.keras.layers.Embedding(context_win, embed_dim, name='pos_emb')

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(seq_len)
        token_emb = self.token_emb(x)
        pos_emb = self.pos_emb(positions)
        # ФИКС: cast scale к dtype embedding (mixed precision: token_emb = float16)
        scale = tf.cast(self.embed_dim, token_emb.dtype)
        return token_emb * tf.math.sqrt(scale) + pos_emb

    def get_config(self):
        config = super().get_config()
        config.update({
            'vocab_size': self.vocab_size,
            'embed_dim': self.embed_dim,
            'context_win': self.context_win,
        })
        return config


print("Perplexity metric + TokenPositionEmbedding — OK")


In [ ]:
# ========================== [CELL 8] TRANSFORMER BLOCK ==========================
# Pre-Norm Transformer decoder block (GPT-2 style).
# V0.24: DROPOUT_RATE=0.20 (was 0.15 in V0.21) — сильнее регуляризация.

class TransformerBlock(tf.keras.layers.Layer):
    """Pre-Norm Transformer Decoder Block."""

    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.att_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ffn_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads,
            dropout=dropout_rate
        )
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='gelu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embed_dim),
            tf.keras.layers.Dropout(dropout_rate),
        ])
        self.drop = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        # Causal mask
        seq_len = tf.shape(x)[1]
        causal_mask = tf.linalg.band_part(
            tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0
        )

        # Pre-Norm Attention
        norm_x = self.att_norm(x)
        att_out = self.att(
            query=norm_x, value=norm_x, key=norm_x,
            attention_mask=causal_mask, training=training
        )
        x = x + self.drop(att_out, training=training)

        # Pre-Norm FFN
        norm_x = self.ffn_norm(x)
        ffn_out = self.ffn(norm_x, training=training)
        x = x + ffn_out

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'embed_dim': self.embed_dim,
            'num_heads': self.num_heads,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate,
        })
        return config


print("TransformerBlock — OK")


In [ ]:
# ========================== [CELL 9] LLM MODEL ==========================
# GPT-2 style decoder-only transformer с weight tying.
# V0.24 changes: float32 logits cast (mixed precision fix).

def build_llm(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_blocks=NUM_BLOCKS,
    context_win=CONTEXT_WIN,
    dropout_rate=DROPOUT_RATE,
):
    inputs = tf.keras.Input(shape=(context_win,), dtype=tf.int32, name='token_ids')

    # Token + Position embedding
    tok_pos_emb = TokenPositionEmbedding(vocab_size, embed_dim, context_win, name='tok_pos_emb')
    x = tok_pos_emb(inputs)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    # Transformer blocks
    for i in range(num_blocks):
        x = TransformerBlock(embed_dim, num_heads, ff_dim, dropout_rate, name=f'block_{i}')(x)

    # Final LayerNorm (Pre-Norm convention)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6, name='final_norm')(x)

    # Weight-tied output: logits = x @ token_emb^T
    token_emb_weights = tok_pos_emb.token_emb.embeddings  # (vocab, embed_dim)
    logits = tf.matmul(x, token_emb_weights, transpose_b=True)  # (B, T, vocab)

    # КРИТИЧНО: cast logits to float32 для численной стабильности (mixed precision)
    logits = tf.cast(logits, tf.float32, name='logits_f32')

    model = tf.keras.Model(inputs=inputs, outputs=logits, name='SimpleLLM_v024')
    return model


model = build_llm()
model.summary(line_length=100, show_trainable=True)

# Подсчёт параметров
total_params = model.count_params()
print(f"\n  Total parameters: {total_params:,}")
print(f"  ≈ {total_params/1e6:.1f}M parameters")


In [ ]:
# ========================== [CELL 10] MODEL SMOKE TEST ==========================
# Быстрая проверка что модель работает до начала обучения.

dummy_input = tf.random.uniform((2, CONTEXT_WIN), minval=0, maxval=VOCAB_SIZE, dtype=tf.int32)
dummy_output = model(dummy_input, training=False)

assert dummy_output.shape == (2, CONTEXT_WIN, VOCAB_SIZE), \
    f"Unexpected shape: {dummy_output.shape}"
assert dummy_output.dtype == tf.float32, \
    f"Logits must be float32 (for numerical stability), got {dummy_output.dtype}"

# Проверяем что logits не NaN/Inf
assert not tf.reduce_any(tf.math.is_nan(dummy_output)), "NaN in logits!"
assert not tf.reduce_any(tf.math.is_inf(dummy_output)), "Inf in logits!"

print("Smoke test PASSED")
print(f"  Input:  {dummy_input.shape} (dtype={dummy_input.dtype})")
print(f"  Output: {dummy_output.shape} (dtype={dummy_output.dtype})")
print(f"  Logits range: [{dummy_output.numpy().min():.4f}, {dummy_output.numpy().max():.4f}]")


In [ ]:
# ========================== [CELL 11] TEXT GENERATION ==========================
# Autoregressive generation с temperature + top-k + top-p sampling.
#
# V0.24: Добавлен repetition_penalty для борьбы с зацикливанием.
# V0.21 fix: RIGHT-pad context, берём logits для позиции last_real_token (не -1).

def generate_text(
    prompt,
    model,
    sp_model,
    max_tokens=150,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.2,  # НОВОЕ V0.24
):
    """Генерирует текст с temperature, top-k, top-p и repetition penalty."""
    # Encode prompt (без EOS — мы продолжаем генерацию)
    ids = sp_model.encode(prompt)
    ids = [sp_model.bos_id()] + ids
    generated = list(ids)

    for _ in range(max_tokens):
        # Truncate/pad to context window (RIGHT-pad)
        if len(generated) >= CONTEXT_WIN:
            ctx = generated[-CONTEXT_WIN:]
            last_pos = CONTEXT_WIN - 1
        else:
            ctx = generated + [sp_model.pad_id()] * (CONTEXT_WIN - len(generated))
            last_pos = len(generated) - 1

        input_ids = tf.constant([ctx], dtype=tf.int32)
        logits = model(input_ids, training=False)  # (1, T, V)
        next_logits = logits[0, last_pos, :]  # (V,)

        # НОВОЕ V0.24: Repetition penalty
        # Уменьшаем вероятность уже сгенерированных токенов
        if repetition_penalty != 1.0:
            unique_generated = set(generated)
            for token_id in unique_generated:
                if next_logits[token_id] > 0:
                    next_logits = tf.tensor_scatter_nd_update(
                        next_logits,
                        [[token_id]],
                        [next_logits[token_id] / repetition_penalty]
                    )
                else:
                    next_logits = tf.tensor_scatter_nd_update(
                        next_logits,
                        [[token_id]],
                        [next_logits[token_id] * repetition_penalty]
                    )

        # Temperature
        if temperature > 0:
            scaled = next_logits / temperature
        else:
            # Greedy
            next_id = tf.argmax(next_logits, axis=-1).numpy()
            generated.append(int(next_id))
            if next_id == sp_model.eos_id():
                break
            continue

        # Top-k filtering
        if top_k > 0:
            top_k_vals, top_k_idx = tf.math.top_k(scaled, k=min(top_k, VOCAB_SIZE))
            mask = tf.fill(scaled.shape, float('-inf'))
            mask = tf.tensor_scatter_nd_update(
                mask,
                tf.expand_dims(top_k_idx, 1),
                top_k_vals
            )
            scaled = mask

        # Top-p (nucleus) filtering
        if 0 < top_p < 1.0:
            sorted_idx = tf.argsort(scaled, direction='DESCENDING')
            sorted_logits = tf.gather(scaled, sorted_idx)
            cumprobs = tf.cumsum(tf.nn.softmax(sorted_logits))
            # Маскируем все после порога
            cutoff = tf.searchsorted(cumprobs, [top_p])[0]
            cutoff = tf.minimum(cutoff + 1, tf.shape(sorted_logits)[0])
            remove_mask = tf.range(tf.shape(sorted_logits)[0]) >= cutoff
            sorted_logits = tf.where(remove_mask, float('-inf'), sorted_logits)
            # Восстанавливаем порядок
            restore_idx = tf.argsort(sorted_idx)
            scaled = tf.gather(sorted_logits, restore_idx)

        # Sample
        next_id = tf.random.categorical(
            tf.expand_dims(scaled, 0), num_samples=1
        )[0, 0].numpy()

        generated.append(int(next_id))
        if next_id == sp_model.eos_id():
            break

    # Decode (убираем BOS)
    clean_ids = [i for i in generated if i not in (sp_model.bos_id(), sp_model.pad_id())]
    return sp_model.decode(clean_ids)


# Quick test (untrained model — garbage expected)
test_out = generate_text("The king said", model, sp_model, max_tokens=30)
print(f"Test generation (untrained): {test_out[:200]}")
print("generate_text — OK")


In [ ]:
# ========================== [CELL 12] TRAINING LOOP ==========================
#
# КЛЮЧЕВЫЕ ИЗМЕНЕНИЯ V0.24 (анти-оверфиттинг):
#
# 1. AdamW с weight_decay=0.01 (L2 регулизация весов)
# 2. Label smoothing = 0.1 (не давать модели быть "уверенной на 100%")
# 3. EarlyStopping patience=3 (быстрее останавливаем при overfit)
# 4. OverfittingDetector callback: мониторит gap между train_loss и val_loss
# 5. Cosine decay с warmup (как V0.21)
# 6. Gradient monitoring callback (как V0.21)

# ---- Learning Rate Schedule: Cosine Decay with Warmup ----
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, total_steps):
        super().__init__()
        self.base_lr = base_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup = tf.minimum(step / tf.maximum(tf.cast(self.warmup_steps, tf.float32), 1.0), 1.0)
        decay_steps = tf.cast(self.total_steps - self.warmup_steps, tf.float32)
        decay_step = tf.maximum(step - tf.cast(self.warmup_steps, tf.float32), 0.0)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * decay_step / tf.maximum(decay_steps, 1.0)))
        return self.base_lr * warmup * cosine

    def get_config(self):
        return {
            'base_lr': self.base_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps,
        }


# ---- Callbacks ----
class GradientMonitor(tf.keras.callbacks.Callback):
    """Мониторит gradient norms — помогает обнаружить exploding/vanishing gradients."""
    def on_epoch_end(self, epoch, logs=None):
        # Подсчитываем среднюю norm весов (как прокси для gradient health)
        norms = []
        for w in self.model.trainable_weights:
            norms.append(tf.norm(w).numpy())
        avg_norm = np.mean(norms)
        max_norm = np.max(norms)
        print(f"  [GradMon] avg_weight_norm={avg_norm:.4f}, max_weight_norm={max_norm:.4f}")


class OverfittingDetector(tf.keras.callbacks.Callback):
    """
    НОВОЕ V0.24: Мониторит разницу между train_loss и val_loss.
    Если gap > threshold на протяжении N эпох — предупреждает.
    """
    def __init__(self, gap_threshold=0.15, patience=3):
        super().__init__()
        self.gap_threshold = gap_threshold
        self.patience = patience
        self.gap_count = 0

    def on_epoch_end(self, epoch, logs=None):
        train_loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        gap = val_loss - train_loss

        if gap > self.gap_threshold:
            self.gap_count += 1
            status = f"WARNING ({self.gap_count}/{self.patience})"
        else:
            self.gap_count = 0
            status = "OK"

        print(f"  [OverfitDetector] train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
              f"gap={gap:.4f}, status={status}")

        if self.gap_count >= self.patience:
            print(f"  *** OVERFITTING DETECTED: gap > {self.gap_threshold} "
                  f"for {self.patience} consecutive epochs! ***")


class GenerationSampler(tf.keras.callbacks.Callback):
    """
    НОВОЕ V0.24: Генерирует примеры каждые N эпох чтобы визуально
    мониторить прогресс и ловить memorization.
    """
    def __init__(self, prompts, sp_model, every_n_epochs=5, max_tokens=60):
        super().__init__()
        self.prompts = prompts
        self.sp_model = sp_model
        self.every_n = every_n_epochs
        self.max_tokens = max_tokens

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n != 0:
            return
        print(f"\n  [GenSampler] Epoch {epoch+1} examples:")
        for prompt in self.prompts:
            try:
                out = generate_text(
                    prompt, self.model, self.sp_model,
                    max_tokens=self.max_tokens, temperature=0.7,
                    top_k=40, top_p=0.9, repetition_penalty=1.2
                )
                # Обрезаем для вывода
                preview = out[:150].replace('\n', ' ')
                print(f"    '{prompt}' -> {preview}")
            except Exception as e:
                print(f"    '{prompt}' -> ERROR: {e}")
        print()


# ---- Schedule & Optimizer ----
steps_per_epoch = len(train_idx) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS

lr_schedule = WarmupCosineDecay(
    base_lr=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    total_steps=total_steps,
)

# V0.24: AdamW с weight_decay для L2 регуляризации
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=WEIGHT_DECAY,
    clipnorm=1.0,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=1e-9,
)

# ---- Loss: label smoothing = 0.1 ----
# SparseCategoricalCrossentropy не поддерживает label_smoothing напрямую,
# поэтому используем custom loss с one-hot conversion.
class LabelSmoothedLoss(tf.keras.losses.Loss):
    """Sparse cross-entropy с label smoothing."""
    def __init__(self, vocab_size, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.ls = label_smoothing

    def call(self, y_true, y_pred):
        # y_true: (B, T) int32,  y_pred: (B, T, V) float32 logits
        one_hot = tf.one_hot(tf.cast(y_true, tf.int32), self.vocab_size)
        smooth = one_hot * (1.0 - self.ls) + self.ls / tf.cast(self.vocab_size, tf.float32)
        # softmax_cross_entropy_with_logits: (B, T)
        loss = tf.nn.softmax_cross_entropy_with_logits(labels=smooth, logits=y_pred)
        return loss  # (B, T) — Keras применит sample_weight автоматически

loss_fn = LabelSmoothedLoss(
    vocab_size=VOCAB_SIZE,
    label_smoothing=LABEL_SMOOTHING,
    reduction='none',      # Keras handles reduction + sample_weight
)

# ---- Compile ----
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    weighted_metrics=[Perplexity()],
)

# ---- Callbacks ----
CHECKPOINT_PATH = 'best_model_v024.weights.h5'

sample_prompts = [
    "The king",
    "Once upon a time",
    "In the darkness",
    "Science tells us",
]

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,  # V0.24: 3 (was 5 in V0.21) — быстрее останавливаем
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    GradientMonitor(),
    OverfittingDetector(gap_threshold=0.15, patience=3),
    GenerationSampler(sample_prompts, sp_model, every_n_epochs=5, max_tokens=60),
]

print(f"Training config:")
print(f"  Optimizer:       AdamW (weight_decay={WEIGHT_DECAY})")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  LR schedule:     Cosine warmup ({WARMUP_STEPS} steps)")
print(f"  Base LR:         {LEARNING_RATE}")
print(f"  Epochs:          {EPOCHS}")
print(f"  EarlyStopping:   patience=3")
print(f"  Steps/epoch:     {steps_per_epoch}")
print(f"  Total steps:     {total_steps}")

# ---- Train! ----
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nTraining complete!")
print(f"  Best val_loss: {min(history.history['val_loss']):.4f}")
print(f"  Best val_perplexity: {min(history.history['val_perplexity']):.2f}")
print(f"  Epochs trained: {len(history.history['loss'])}")


In [ ]:
# ========================== [CELL 13] VALIDATION EXAMPLES ==========================
# Генерируем текст из разных доменов корпуса + off-domain prompts.
# Проверяем: (1) разнообразие стилей, (2) нет verbatim memorization, (3) нет Bible bias.

validation_prompts = [
    # Литература (разные стили)
    "The king ordered his men to",
    "Once upon a time, in a land far away",
    "She walked through the dark forest",
    "The ship sailed across the",

    # Научный/философский стиль
    "The nature of truth is",
    "Science has shown that",

    # Off-domain (модель НЕ видела этих тем — тест на generalization)
    "The computer program was",
    "In the year 2024",
    "The recipe for chocolate cake",

    # Тест на Bible bias (если модель всё ещё показывает Bible стиль — overfit)
    "And he said unto",
    "The son of",
]

print("=" * 70)
print("  VALIDATION GENERATION (V0.24)")
print("  Ищем: разнообразие стилей, отсутствие verbatim Bible quotes,")
print("  минимум Gutenberg license leaks")
print("=" * 70)

for prompt in validation_prompts:
    print(f"\n  PROMPT: '{prompt}'")
    for temp in [0.7, 1.0]:
        text = generate_text(
            prompt, model, sp_model,
            max_tokens=100, temperature=temp,
            top_k=40, top_p=0.9, repetition_penalty=1.2,
        )
        label = f"T={temp}"
        print(f"    [{label}] {text[:200]}")
    print("-" * 60)

# Memorization check: ищем подозрительные паттерны
print("\n" + "=" * 70)
print("  MEMORIZATION CHECK")
print("=" * 70)
suspicious_patterns = [
    'paragraph 1.', 'full refund', 'electronic work', 'gutenberg',
    'project gutenberg', 'trademark', '501(c)', 'redistribute',
]
all_generated = []
for prompt in validation_prompts:
    text = generate_text(prompt, model, sp_model, max_tokens=150,
                         temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2)
    all_generated.append(text)

for pattern in suspicious_patterns:
    found = [g for g in all_generated if pattern.lower() in g.lower()]
    status = f"FOUND in {len(found)} generations!" if found else "clean"
    print(f"  '{pattern}': {status}")


In [ ]:
# ========================== [CELL 14] INTERACTIVE GENERATION ==========================
# Интерактивный цикл генерации текста.
# Поддерживает команды: /temp X, /topk X, /topp X, /tokens X, /rep X, /quit

print("=" * 60)
print("  SimpleLLM V0.24 — Interactive Generation")
print("  Команды: /temp N  /topk N  /topp N  /tokens N  /rep N  /quit")
print("=" * 60)

gen_temperature = 0.8
gen_top_k = 40
gen_top_p = 0.9
gen_max_tokens = 150
gen_rep_penalty = 1.2

while True:
    try:
        prompt = input("\nPrompt> ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        break

    if not prompt:
        continue
    if prompt.lower() == '/quit':
        print("Bye!")
        break

    # Parse commands
    if prompt.startswith('/temp '):
        gen_temperature = float(prompt.split()[1])
        print(f"  Temperature set to {gen_temperature}")
        continue
    if prompt.startswith('/topk '):
        gen_top_k = int(prompt.split()[1])
        print(f"  Top-k set to {gen_top_k}")
        continue
    if prompt.startswith('/topp '):
        gen_top_p = float(prompt.split()[1])
        print(f"  Top-p set to {gen_top_p}")
        continue
    if prompt.startswith('/tokens '):
        gen_max_tokens = int(prompt.split()[1])
        print(f"  Max tokens set to {gen_max_tokens}")
        continue
    if prompt.startswith('/rep '):
        gen_rep_penalty = float(prompt.split()[1])
        print(f"  Repetition penalty set to {gen_rep_penalty}")
        continue

    # Generate
    output = generate_text(
        prompt, model, sp_model,
        max_tokens=gen_max_tokens,
        temperature=gen_temperature,
        top_k=gen_top_k,
        top_p=gen_top_p,
        repetition_penalty=gen_rep_penalty,
    )
    print(f"\n{output}")
